In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 01 — Ingestão Bronze
# MAGIC
# MAGIC **Objetivo desta etapa:** trazer os arquivos brutos (`application_train.csv`, `bureau.csv`)
# MAGIC para o ambiente de nuvem exatamente como foram recebidos, preservando rastreabilidade.
# MAGIC
# MAGIC Fonte: Home Credit Default Risk (Kaggle). Licença: <preencher>.

# COMMAND ----------

# MAGIC %md
# MAGIC ### Configuração de catálogo e volume
# MAGIC Ajuste os nomes conforme seu ambiente Databricks (Unity Catalog).

# COMMAND ----------

CATALOG = "mvp_credito"
VOLUME_PATH = "/Volumes/mvp_credito/dados_credito/dados/"  # onde os CSVs originais foram enviados

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.bronze")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.gold")

# COMMAND ----------

# MAGIC %md
# MAGIC ### Leitura dos CSVs brutos
# MAGIC Nenhuma transformação de conteúdo é feita aqui — apenas leitura e adição de metadados de controle.

# COMMAND ----------

from pyspark.sql import functions as F

def ler_csv_bruto(caminho_arquivo: str, nome_fonte: str):
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)  # na Bronze aceitamos inferência simples; tipagem final é feita na Silver
        .csv(caminho_arquivo)
    )
    df = (
        df
        .withColumn("_ingestion_date", F.current_timestamp())
        .withColumn("_source_file", F.lit(nome_fonte))
    )
    return df

df_application_raw = ler_csv_bruto(f"{VOLUME_PATH}/application_train.csv", "application_train.csv")
df_bureau_raw = ler_csv_bruto(f"{VOLUME_PATH}/bureau.csv", "bureau.csv")

# COMMAND ----------

# MAGIC %md
# MAGIC ### Persistência das tabelas Bronze (Delta)

# COMMAND ----------

df_application_raw.write.mode("overwrite").saveAsTable(f"{CATALOG}.bronze.application_train_raw")
df_bureau_raw.write.mode("overwrite").saveAsTable(f"{CATALOG}.bronze.bureau_raw")

# COMMAND ----------

# MAGIC %md
# MAGIC ### Verificação
# MAGIC Confirmar volume de registros e schema recebido (tirar screenshot desta célula para o README).

# COMMAND ----------

print("application_train_raw:", spark.table(f"{CATALOG}.bronze.application_train_raw").count(), "linhas")
print("bureau_raw:", spark.table(f"{CATALOG}.bronze.bureau_raw").count(), "linhas")

display(spark.table(f"{CATALOG}.bronze.application_train_raw").limit(5))


application_train_raw: 307511 linhas
bureau_raw: 1716428 linhas


SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,OWN_CAR_AGE,FLAG_MOBIL,FLAG_EMP_PHONE,FLAG_WORK_PHONE,FLAG_CONT_MOBILE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,REGION_RATING_CLIENT,REGION_RATING_CLIENT_W_CITY,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,REG_REGION_NOT_LIVE_REGION,REG_REGION_NOT_WORK_REGION,LIVE_REGION_NOT_WORK_REGION,REG_CITY_NOT_LIVE_CITY,REG_CITY_NOT_WORK_CITY,LIVE_CITY_NOT_WORK_CITY,ORGANIZATION_TYPE,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,APARTMENTS_AVG,BASEMENTAREA_AVG,YEARS_BEGINEXPLUATATION_AVG,YEARS_BUILD_AVG,COMMONAREA_AVG,ELEVATORS_AVG,ENTRANCES_AVG,FLOORSMAX_AVG,FLOORSMIN_AVG,LANDAREA_AVG,LIVINGAPARTMENTS_AVG,LIVINGAREA_AVG,NONLIVINGAPARTMENTS_AVG,NONLIVINGAREA_AVG,APARTMENTS_MODE,BASEMENTAREA_MODE,YEARS_BEGINEXPLUATATION_MODE,YEARS_BUILD_MODE,COMMONAREA_MODE,ELEVATORS_MODE,ENTRANCES_MODE,FLOORSMAX_MODE,FLOORSMIN_MODE,LANDAREA_MODE,LIVINGAPARTMENTS_MODE,LIVINGAREA_MODE,NONLIVINGAPARTMENTS_MODE,NONLIVINGAREA_MODE,APARTMENTS_MEDI,BASEMENTAREA_MEDI,YEARS_BEGINEXPLUATATION_MEDI,YEARS_BUILD_MEDI,COMMONAREA_MEDI,ELEVATORS_MEDI,ENTRANCES_MEDI,FLOORSMAX_MEDI,FLOORSMIN_MEDI,LANDAREA_MEDI,LIVINGAPARTMENTS_MEDI,LIVINGAREA_MEDI,NONLIVINGAPARTMENTS_MEDI,NONLIVINGAREA_MEDI,FONDKAPREMONT_MODE,HOUSETYPE_MODE,TOTALAREA_MODE,WALLSMATERIAL_MODE,EMERGENCYSTATE_MODE,OBS_30_CNT_SOCIAL_CIRCLE,DEF_30_CNT_SOCIAL_CIRCLE,OBS_60_CNT_SOCIAL_CIRCLE,DEF_60_CNT_SOCIAL_CIRCLE,DAYS_LAST_PHONE_CHANGE,FLAG_DOCUMENT_2,FLAG_DOCUMENT_3,FLAG_DOCUMENT_4,FLAG_DOCUMENT_5,FLAG_DOCUMENT_6,FLAG_DOCUMENT_7,FLAG_DOCUMENT_8,FLAG_DOCUMENT_9,FLAG_DOCUMENT_10,FLAG_DOCUMENT_11,FLAG_DOCUMENT_12,FLAG_DOCUMENT_13,FLAG_DOCUMENT_14,FLAG_DOCUMENT_15,FLAG_DOCUMENT_16,FLAG_DOCUMENT_17,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR,_ingestion_date,_source_file
387559,0,Cash loans,F,Y,Y,0,247500.0,896602.5,32202.0,774000.0,Family,Commercial associate,Secondary / secondary special,Married,House / apartment,0.00702,-16713,-1709,-8453.0,-253,4.0,1,1,0,1,0,0,Medicine staff,2.0,2,2,SATURDAY,14,0,0,0,0,0,0,Medicine,0.5598017718965242,0.6622635801794581,null,0.1113,null,0.9896,null,null,0.2,0.1724,0.3333,null,null,null,0.1085,null,null,0.1134,null,0.9896,null,null,0.2014,0.1724,0.3333,null,null,null,0.113,null,null,0.1124,null,0.9896,null,null,0.2,0.1724,0.3333,null,null,null,0.1104,null,null,null,block of flats,0.1067,Panel,No,1.0,1.0,1.0,1.0,-313.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,null,null,null,null,null,null,2026-09-22T23:57:40.922Z,application_train.csv
387560,0,Cash loans,F,N,Y,0,126000.0,225000.0,11619.0,225000.0,Family,Pensioner,Secondary / secondary special,Civil marriage,House / apartment,0.026392000000000002,-23611,365243,-3878.0,-4265,null,1,0,0,1,0,0,null,2.0,2,2,THURSDAY,12,0,0,0,0,0,0,XNA,null,0.4456307526614687,0.1385128770585923,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,0.0,0.0,0.0,0.0,0.0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,1.0,0.0,2026-09-22T23:57:40.922Z,application_train.csv
387561,0,Cash loans,F,N,N,0,103500.0,677664.0,22527.0,585000.0,Unaccompanied,Pensioner,Secondary / secondary special,Married,House / apartment,0.025164,-20980,365243,-6833.0,-4513,null,1,0,0,1,0,0,null,2.0,2,2,WEDNESDAY,9,0,0,0,0,0,0,XNA,0.8717818747562379,0.7345035993730815,0.8193176922872417,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,